# f696 Stage1 原生来源：运行时与父本一致性预检

只读封存的 Qlib Recorder 与原价来源，在仓库虚拟环境中核对 MLflow 文件库访问、完整 f696 A0 五输出和 944 个同日日期。没有新拟合、候选账户或收益。

In [1]:
from pathlib import Path
import hashlib,json,os
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"]="1"
import numpy as np
import pandas as pd
import qlib,sklearn
from qlib.data import D
from qlib.workflow import R
from src.workflow.stage1_nvda_guidance_receiver import prepare_account_sources
root=Path.cwd();card=root/"research/stage1_f696_native_source_runtime_preflight_20261023"
plan=json.loads((card/"plan.json").read_text());src=plan["sources"]
sha=lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert os.environ["MLFLOW_ALLOW_FILE_STORE"]=="true"
assert qlib.__version__==plan["runtime"]["expected_qlib_version"]
assert sklearn.__version__==plan["runtime"]["expected_sklearn_version"]
helper=root/src["helper_card"]
assert sha(helper/"account_source_preflight.json")==src["helper_receipt_sha256"]
account=prepare_account_sources(root=root,card=helper)
assert sha(helper/"account_source_preflight.json")==src["helper_receipt_sha256"]
A_card=root/src["f696_card"]
assert sha(A_card/"summary.json")==src["f696_summary_sha256"]
A_path=A_card/"qlib_recorder"/src["f696_experiment_id"]/src["f696_recorder_id"]/"artifacts/inputs/frame"
assert sha(A_path)==src["f696_frame_sha256"]
with R.uri_context(str((A_card/"qlib_recorder").resolve())):
    by_id=R.get_recorder(experiment_id=src["f696_experiment_id"],recorder_id=src["f696_recorder_id"])
    by_name=R.get_recorder(experiment_name=A_card.name,recorder_id=src["f696_recorder_id"])
    assert by_id.id==by_name.id==src["f696_recorder_id"]
    A_frame=by_id.load_object("inputs/frame")
cols=plan["fixed_check"]["columns"]
dates=["signal_date","trade_date","exit_date"]
raw=account["frame"].reset_index(drop=True)
target=account["source_targets"].copy()
A_frame=A_frame.reset_index(drop=True)
assert len(raw)==len(target)==len(A_frame)==plan["fixed_check"]["rows"]
pd.testing.assert_frame_equal(raw[dates],A_frame[dates])
pd.testing.assert_index_equal(pd.DatetimeIndex(target.index),pd.DatetimeIndex(raw.signal_date),check_names=False)
raw_diff=np.abs(raw[cols].to_numpy(float)-A_frame[cols].to_numpy(float))
raw_mismatch=int((raw_diff.max(axis=1)>1e-8).sum())
assert raw_mismatch==plan["fixed_check"]["original_raw_frame_mismatch_rows"]
corrected=raw.copy(deep=True)
for col in cols:corrected[col]=target[col].to_numpy(copy=True)
pd.testing.assert_frame_equal(corrected[dates],A_frame[dates])
corrected_diff=np.abs(corrected[cols].to_numpy(float)-A_frame[cols].to_numpy(float))
assert float(corrected_diff.max())==plan["fixed_check"]["corrected_vs_original_A_max_abs_error"]
for role in plan["fixed_check"]["reference_native_audits"]:
    assert account["references"][role]["audit"]["status"]=="passed"
assert sha(root/src["provider_uri"]/"qlib_store_manifest.json")==src["provider_manifest_sha256"]
qlib.init(provider_uri=str((root/src["provider_uri"]).resolve()),region="us",expression_cache=None,dataset_cache=None)
calendar=pd.DatetimeIndex(D.calendar(start_time="2011-01-03",end_time="2026-10-09",freq="day"))
close=D.features(["SOXX"],["$close"],start_time="2011-01-03",end_time="2026-10-09",freq="day")["$close"].droplevel("instrument")
assert close.index.equals(calendar) and close.reindex(pd.DatetimeIndex(raw.signal_date)).notna().all()
assert pd.DatetimeIndex(raw.signal_date).isin(calendar).all()
result={"experiment":card.name,"status":"qlib_venv_file_store_runtime_and_exact_f696_source_preflight_passed",
        "plan_sha256":sha(card/"plan.json"),"runtime":{"qlib":qlib.__version__,"sklearn":sklearn.__version__,
        "MLFLOW_ALLOW_FILE_STORE":os.environ["MLFLOW_ALLOW_FILE_STORE"]},
        "source_helper_receipt_sha256":src["helper_receipt_sha256"],
        "f696_recorder_id":src["f696_recorder_id"],"f696_get_recorder_by_id_and_name":True,
        "rows":len(raw),"raw_27baa_vs_f696_mismatch_rows":raw_mismatch,
        "raw_vs_f696_max_abs_difference":float(raw_diff.max()),
        "corrected_all_five_outputs_max_abs_difference":float(corrected_diff.max()),
        "SOXX_and_Stage1_A_native_audit_passed":True,
        "QLib_calendar_rows":len(calendar),"all_signal_dates_in_Qlib_calendar":True,
        "new_fits":0,"new_accounts":0,"new_Recorders":0,"new_bootstraps":0,
        "economic_result":None,"stopped_static_composition_plan_reopened":False}
(card/"result.json").write_text(json.dumps(result,ensure_ascii=False,indent=2)+"\n")
summary={"experiment":card.name,"decision":"runtime_preflight_pass_no_economic_candidate",
         "objective_achieved":False,"plan_sha256":result["plan_sha256"],"result":"result.json",
         "new_model_fits":0,"new_native_accounts":0,
         "artifact_policy":{"policy":"slim_card_recorder_first","workflow_config":"workflow.yaml",
                            "recorder_only_artifacts":[],
                            "reused_native_sources":[src["helper_card"],src["f696_card"]]},
         "limitations":["No candidate policy, account or benchmark comparison was executed.",
                        "The corrected source frame is a preflight object and was not saved as a new allocation proposal or release."]}
(card/"summary.json").write_text(json.dumps(summary,ensure_ascii=False,indent=2)+"\n")
(card/"workflow.yaml").write_text("experiment: "+card.name+"\nentrypoint: "+card.name+".ipynb\nphase: read_only_runtime_preflight\n")
(card/"conclusion.md").write_text(
 "# f696 Stage1 原生来源加载：运行条件与父本逐值校验\n\n"
 "仓库 `qlib_venv` 的 Qlib0.9.7/Scikit-learn1.9.1 在导入旧MLflow文件Recorder前须设置 `MLFLOW_ALLOW_FILE_STORE=true`。"
 "此前的 `No valid experiment has been found` 是缺少此运行条件，不是已封存实验ID或来源缺失。"
 "设置后，f696 Recorder按名称和ID均能取得，helper载入944行，原Stage1 A0和SOXX原生审计通过。\n\n"
 f"helper原27baa五输出与f696原A0提议 {raw_mismatch}/944 日期不一致；逐行替换五项`frozen_current`后，"
 f"对封存A0的最大绝对误差 {corrected_diff.max():.1f}。所有信号日均在钉住的Qlib日历且SOXX收盘价可读。"
 "本卡0拟合、0账户、0新Recorder；已停止的静态组成重测并未重开。"
 "下一真正候选仍须从f696失败区间得出独立决策时机制，并运行完整0bp Tiered自身状态账户。\n")
print(json.dumps({"status":result["status"],"raw_mismatch":raw_mismatch,"corrected_max":float(corrected_diff.max()),"new_accounts":0}))

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[80590:MainThread](2026-10-23 23:36:39,295) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[80590:MainThread](2026-10-23 23:36:39,296) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[80590:MainThread](2026-10-23 23:36:39,296) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/core_1min')}


[80590:MainThread](2026-10-23 23:36:40,499) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[80590:MainThread](2026-10-23 23:36:40,500) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[80590:MainThread](2026-10-23 23:36:40,500) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


{"status": "qlib_venv_file_store_runtime_and_exact_f696_source_preflight_passed", "raw_mismatch": 944, "corrected_max": 0.0, "new_accounts": 0}


## 中文结论

运行条件、原始27baa与f696差异、替换后逐值校验及原生来源审计见 `result.json`、`conclusion.md`。这只是可重复的来源预检；不会重启已停止的静态组成方案，也不代表超过SOXX。